<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/07_fine_tuning_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Gemma 3 1B-IT for Sentiment Analysis

In this notebook, we fine-tune the **Gemma 3 1B** model on our enriched and augmented financial sentiment dataset. We use **QLoRA (4-bit quantization + LoRA)** to make the process efficient enough for consumer GPUs.

### Why Fine-tune with Explanations?
- **Reasoning Injection**: By including the `explanation` column in our training prompts, we teach the model to "think" about the financial implications before deciding on a sentiment label.
- **Small Model, High Performance**: Fine-tuning allows a 1B parameter model to outperform much larger general-purpose models on specialized tasks like financial analysis.

### Technical Approach
- **Library**: Hugging Face `trl` and `peft`.
- **Method**: Supervised Fine-Tuning (SFT) with LoRA.
- **Format**: Chat-style prompts using the Gemma 3 template.

In [ ]:
%%capture
import os
if "COLAB_" in "".join(os.environ.keys()):
    !pip install -U transformers trl accelerate bitsandbytes

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, BitsAndBytesConfig
from transformers.models.gemma3 import Gemma3ForCausalLM
from datasets import load_from_disk, Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import warnings

warnings.filterwarnings("ignore")

## 1. Configuration

In [ ]:
GEMMA_PATH = "google/gemma-3-1b-it"
DATASET_PATH = "FinancialPhraseBank_explained"
OUTPUT_DIR = "gemma-3-1b-sentiment-lora"
DEMO = False

## 2. Loading the Dataset

We load our locally saved dataset from the previous step. This dataset contains both original and augmented examples, each with a reasoning explanation.

In [ ]:
ds = load_from_disk(DATASET_PATH)
print(ds)

def rename_cols(example):
    return {
        "text": example["sentence"],
        "sentiment": example["sentiment"],
        "reasoning": example["explanation"]
    }

ds = ds.map(rename_cols)

## 3. Prompt Engineering

We use a specific instruction template to guide the model's output format.

In [ ]:
INSTRUCTION = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)

tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_train_prompt(example):
    messages = [
        {"role": "user", "content": INSTRUCTION + f'Headline: "{example["text"]}"'},
        {"role": "assistant", "content": f"<sentiment>{example['sentiment']}</sentiment>\n<reasoning>{example['reasoning']}</reasoning>"}
    ]
    return {"prompt_text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_ds = ds["train"].map(generate_train_prompt)
eval_ds = ds["validation"].map(generate_train_prompt)

## 4. Fine-Tuning with QLoRA

We set up the trainer with LoRA adapters targeting all linear layers for maximum expressiveness with minimal parameter count.

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

model = Gemma3ForCausalLM.from_pretrained(
    GEMMA_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    ),
    device_map="auto",
)

peft_config = LoraConfig(
    r=8, lora_alpha=32, target_modules="all-linear", lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="prompt_text",
    max_seq_length=1024,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=eval_ds, peft_config=peft_config, args=training_args)
trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)